# Solution 2.2: Data Types and Subsetting (Angola IEA)

This notebook continues with the Q4 2025 IEA file. Using only the tools from
Lesson 2.2, you fix the data types and save a **typed checkpoint** that the
cleaning step (2.3) picks up.

You will practice:
- Telling a DataFrame from a Series, and checking dtypes
- Renaming 29 Portuguese variable names to readable English ones
- Keeping identifiers as text, including the float to int to string route
- Converting a YYYYMMDD number into a real date with `pd.to_datetime()`
- Saving memory with the `category` dtype
- Subsetting to inspect problems, not yet to fix them
- Saving a typed checkpoint to `10_cleaned/`

> **Pipeline:** reads `0_raw/`, writes `10_cleaned/angola_iea_2025q4_typed.csv`.
> Exercise 2.3 reads that file.

### Path Setup (run first)

In [1]:
import os

import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola/employment_survey'
DATA_CLEAN_DIR = '../../data/10_cleaned'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'
raw_path = os.path.join(DATA_RAW_DIR, RAW_FILE)

SPSS_COLS = [
    'NIDF', 'PPNO', 'G_06_ID_IEA', 'PROV', 'AREA_RESID', 'G_15_TRIMESTRE',
    'DEM_REL', 'DEM_SEX', 'DEM_AGE', 'DEM_MRT', 'DEM_EDL', 'S03_01',
    'ATW_PAY', 'ATW_PFT', 'ATW_FAM', 'ABS_JOB',
    'SRH_JOB', 'SRH_BUS', 'SRH_AVN', 'SRH_AVL', 'SRH_DES',
    'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'MJJ_EMP_REL', 'GHVEDT',
    'POND_IEA_IV_TRIM_2025_IND', 'G_12', 'G_13',
]

df = pd.read_spss(raw_path, usecols=SPSS_COLS, convert_categoricals=False)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

Loaded: (53353, 29)


,NIDF,G_06_ID_IEA,PROV,DEM_REL,AREA_RESID,G_12,G_13,G_15_TRIMESTRE,DEM_SEX,PPNO,...,MJT_SYR,WKT_USHRSTOT,WKT_ACHRSTOT,SRH_JOB,SRH_BUS,SRH_DES,SRH_AVN,SRH_AVL,GHVEDT,POND_IEA_IV_TRIM_2025_IND
0,"9,250,068.00",925.00,24.00,2.00,1.00,NaN,NaN,4.00,2.00,2.00,...,"2,022.00",66.00,66.00,NaN,NaN,NaN,NaN,NaN,"20,251,204.00",322.02
1,"12,870,026.00","1,287.00",30.00,5.00,2.00,NaN,NaN,4.00,1.00,9.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,171.30
2,"4,600,048.00",460.00,16.00,3.00,2.00,NaN,NaN,4.00,1.00,2.00,...,NaN,NaN,NaN,2.00,2.00,1.00,1.00,NaN,"20,251,212.00","1,010.98"
3,"4,600,023.00",460.00,16.00,1.00,2.00,NaN,NaN,4.00,1.00,1.00,...,NaN,NaN,NaN,2.00,2.00,2.00,NaN,NaN,"20,251,212.00",999.80
4,"4,790,052.00",479.00,16.00,1.00,2.00,NaN,NaN,4.00,1.00,1.00,...,"1,998.00",30.00,30.00,NaN,NaN,NaN,NaN,NaN,"20,251,216.00","1,011.64"


---

## Task 1: DataFrame vs Series

Selecting one column returns a **Series**, a one dimensional object with its own
dtype and its own methods. `.str`, `.dt` and `.value_counts()` all belong to
Series, not to the whole DataFrame.

In [2]:
print(type(df))
print(type(df['DEM_AGE']))

<class 'pandas.DataFrame'>
<class 'pandas.Series'>


In [3]:
df.dtypes

NIDF                         float64
G_06_ID_IEA                  float64
PROV                         float64
DEM_REL                      float64
AREA_RESID                   float64
G_12                         float64
G_13                         float64
G_15_TRIMESTRE               float64
DEM_SEX                      float64
PPNO                         float64
DEM_AGE                      float64
DEM_MRT                      float64
S03_01                       float64
DEM_EDL                      float64
ATW_PAY                      float64
ATW_PFT                      float64
ATW_FAM                      float64
ABS_JOB                      float64
MJJ_EMP_REL                  float64
MJT_SYR                      float64
WKT_USHRSTOT                 float64
WKT_ACHRSTOT                 float64
SRH_JOB                      float64
SRH_BUS                      float64
SRH_DES                      float64
SRH_AVN                      float64
SRH_AVL                      float64
G

**Answers:**

- Every one of the 29 columns is `float64`, because SPSS stores everything
  numerically and we asked for raw codes.
- `NIDF` is a household identifier, so `float64` is wrong: identifiers are labels,
  not quantities.
- `GHVEDT` is a date stored as the number 20251204. Nothing date-like works on it
  until it is converted.

---

## Task 2: Rename the columns

The SPSS names come from the questionnaire, not from the analysis. `PROV` and
`ATW_PAY` are precise but unreadable. Rename once, here, and every later notebook
is easier to follow.

In [4]:
RENAME_MAP = {
    'NIDF': 'household_id', 'PPNO': 'person_no', 'G_06_ID_IEA': 'cluster_id',
    'PROV': 'province_code', 'AREA_RESID': 'area_type', 'G_15_TRIMESTRE': 'quarter',
    'DEM_REL': 'rel_to_head', 'DEM_SEX': 'sex', 'DEM_AGE': 'age',
    'DEM_MRT': 'marital_status', 'DEM_EDL': 'education_level',
    'S03_01': 'school_attendance', 'ATW_PAY': 'worked_for_pay',
    'ATW_PFT': 'worked_own_account', 'ATW_FAM': 'worked_family_business',
    'ABS_JOB': 'absent_from_job', 'SRH_JOB': 'sought_work',
    'SRH_BUS': 'sought_business', 'SRH_AVN': 'available_now',
    'SRH_AVL': 'available_2wk', 'SRH_DES': 'wants_work',
    'WKT_USHRSTOT': 'hours_usual', 'WKT_ACHRSTOT': 'hours_actual',
    'MJT_SYR': 'job_start_year', 'MJJ_EMP_REL': 'employment_relation',
    'GHVEDT': 'interview_date', 'POND_IEA_IV_TRIM_2025_IND': 'weight_ind',
    'G_12': 'hh_size_reported', 'G_13': 'hh_adults_reported',
}

df = df.rename(columns=RENAME_MAP)
print(df.columns.tolist())

['household_id', 'cluster_id', 'province_code', 'rel_to_head', 'area_type', 'hh_size_reported', 'hh_adults_reported', 'quarter', 'sex', 'person_no', 'age', 'marital_status', 'school_attendance', 'education_level', 'worked_for_pay', 'worked_own_account', 'worked_family_business', 'absent_from_job', 'employment_relation', 'job_start_year', 'hours_usual', 'hours_actual', 'sought_work', 'sought_business', 'wants_work', 'available_now', 'available_2wk', 'interview_date', 'weight_ind']


**Answers:**

- All 29 names changed. `rename` only touches keys it finds, so a typo in the
  dictionary fails silently: the old name simply survives. Printing the result is
  the check.
- `available_now` and `available_2wk` are deliberately distinguished. They are two
  different questions and 2.4 needs both.

---

## Task 3: Keep identifiers as text

`household_id` arrives as `9250068.0`. Converting straight to string would keep
the `.0`, so the route is float to integer to string.

`province_code` gets the same treatment plus `zfill(2)`. Angola's province codes
run 10 to 30, so `zfill(2)` changes nothing today. It is a documented safeguard,
the same way you would pad any code that could gain a single digit value later.

In [5]:
print('Before:', df['household_id'].head(3).tolist())

for col in ['household_id', 'person_no', 'cluster_id']:
    df[col] = df[col].astype('int64').astype('string')

df['province_code'] = df['province_code'].astype('int64').astype('string').str.zfill(2)

print('After: ', df['household_id'].head(3).tolist())
print('Provinces:', sorted(df['province_code'].unique()))

Before: [9250068.0, 12870026.0, 4600048.0]
After:  ['9250068', '12870026', '4600048']
Provinces: ['10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30']


**Answers:**

- Without the `int64` step you get `'9250068.0'`, which will not join to anything.
- The codes are `'10'` through `'30'`, 21 provinces. `zfill(2)` is a no-op on this
  file and that is fine: it documents the intent and protects a future extract.
- You never add or average an identifier, so storing it as a number invites
  exactly one kind of bug and prevents none.

---

## Task 4: Convert the interview date

`interview_date` is the float `20251204.0`, meaning 2025-12-04. Convert through
`Int64` (which tolerates the missing values) to string, then parse with an
explicit format.

In [6]:
print('Raw values:', df['interview_date'].dropna().head(3).tolist())

date_text = df['interview_date'].astype('Int64').astype('string')
df['interview_date'] = pd.to_datetime(date_text, format='%Y%m%d', errors='raise')

print('dtype:', df['interview_date'].dtype)
print('Range:', df['interview_date'].min(), 'to', df['interview_date'].max())
print('Missing (NaT):', df['interview_date'].isna().sum())

Raw values: [20251204.0, 20251212.0, 20251212.0]
dtype: datetime64[us]
Range: 2024-11-10 00:00:00 to 2026-01-28 00:00:00
Missing (NaT): 23671


In [7]:
# The .dt accessor unlocks date parts
print(df['interview_date'].dt.month.value_counts(dropna=False).sort_index())

interview_date
1.00       553
11.00     9029
12.00    20100
NaN      23671
Name: count, dtype: int64


**Answers:**

- The range is **2024-11-10 to 2026-01-28**. For a survey labelled 4th quarter
  2025, both ends are impossible: they fall outside October to December 2025. The
  month counts show 9,026 interviews in November 2025 and 20,100 in December, 553
  in January 2026 and 3 in November 2024. Not one interview is dated October,
  which for a quarter that officially runs October to December is a finding in
  itself and worth asking the data producer about. This is a genuine data quality
  finding to report, and 2.3 adds a rule for it.
- 23,671 rows have `NaT`. The date is only recorded for the labour module
  respondents, not for every household member.
- `NaT` is the datetime equivalent of `NaN`. Arithmetic on it propagates rather
  than raising.

---

## Task 5: Save memory with `category`

`area_type` holds two distinct values repeated 53,353 times. The `category` dtype
stores each label once and keeps small integer codes alongside.

In [8]:
before = df['area_type'].memory_usage(deep=True)
after = df['area_type'].astype('category').memory_usage(deep=True)

print(f'float64:  {before:,} bytes')
print(f'category: {after:,} bytes')
print(f'Saved:    {(1 - after / before) * 100:.1f}%')

float64:  426,956 bytes
category: 53,501 bytes
Saved:    87.5%


**Answers:**

- 426,956 bytes down to 53,609, an 87% saving on that one column.
- We do **not** persist the category conversion here. The checkpoint is a CSV, and
  CSV has no dtype system, so the saving would be thrown away on the next read.
  It is worth doing in memory on a wide file.

---

## Task 6: Subset to inspect the problems

Filtering here is for **looking**, not fixing. 2.3 makes the removal decisions.

In [9]:
# Implausible working weeks
print('hours_usual > 100:', (df['hours_usual'] > 100).sum())
df[df['hours_usual'] > 100][['household_id', 'hours_usual', 'hours_actual']].head()

hours_usual > 100: 45


,household_id,hours_usual,hours_actual
718,9580039,108.00,108.00
3304,6030115,105.00,105.00
3362,6100150,105.00,105.00
3441,6220127,105.00,105.00
5160,5070064,997.00,997.00


In [10]:
# Combine conditions: each one needs its own parentheses
old_and_working = df[(df['age'] >= 65) & (df['hours_usual'] > 40)]
print('People 65+ working over 40 hours:', len(old_and_working))
old_and_working[['household_id', 'age', 'hours_usual']].head()

People 65+ working over 40 hours: 140


,household_id,age,hours_usual
401,1610102,67.00,42.00
1422,11540028,70.00,48.00
1423,11560160,75.00,60.00
1428,11680111,66.00,70.00
1432,11820156,67.00,60.00


In [11]:
# isin() for a set of provinces: Luanda and Benguela
target = df[df['province_code'].isin(['14', '23'])]
print('Rows in Luanda or Benguela:', len(target))

# str.contains() is safe with missing values when na=False
print('Codes containing "1":', df[df['province_code'].str.contains('1', na=False)]['province_code'].nunique())

Rows in Luanda or Benguela: 6900
Codes containing "1": 11


**Answers:**

- 45 people report more than 100 usual hours a week. Of those, 3 carry the `997`
  sentinel, not yet recoded at this stage, leaving 42 with a genuine implausible
  report of over 14 hours every day with no day off. Not impossible to record,
  but implausible enough for a rule.
- 6,900 rows fall in Luanda or Benguela.
- Each condition needs parentheses because `&` binds tighter than `>` in Python,
  so `df['age'] >= 65 & df['hours_usual'] > 40` parses in the wrong order and
  raises.
- `na=False` makes a missing value count as "does not match" instead of
  propagating `NaN` into the mask, which would raise on indexing.

---

## Task 7: Save the typed checkpoint

Types are fixed. Save so 2.3 starts from a stable baseline.

> Never write into `0_raw/`. This goes to `10_cleaned/`.

In [12]:
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_typed.csv')

df.to_csv(out_path, index=False)
print('Saved:', out_path, '|', df.shape)

Saved: ../../data/10_cleaned/angola_iea_2025q4_typed.csv | (53353, 29)


In [13]:
check = pd.read_csv(out_path, dtype={
    'household_id': 'string', 'person_no': 'string',
    'cluster_id': 'string', 'province_code': 'string',
})
print('Reloaded:', check.shape)
print('interview_date dtype after reload:', check['interview_date'].dtype)
check[['household_id', 'province_code', 'interview_date']].head()

Reloaded: (53353, 29)
interview_date dtype after reload: str


,household_id,province_code,interview_date
0,9250068,24,2025-12-04
1,12870026,30,NaN
2,4600048,16,2025-12-12
3,4600023,16,2025-12-12
4,4790052,16,2025-12-16


**Answers:**

- The checkpoint is 53,353 rows by 29 columns. Nothing has been removed yet: this
  step fixed types only.
- `interview_date` reloads as `object`, plain text. CSV cannot store a datetime,
  so every notebook that reads this file must parse it again. That is the price
  of a portable format, and it is why the `dtype=` argument is needed for the
  identifier columns too.
- `index=False` stops pandas writing the row numbers as a nameless first column,
  which would reappear as `Unnamed: 0` on the next read.